[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day9_live.ipynb)

# Day 9 · 강의 — 에이전트의 구조

설비 일지를 대신 찾아 주는 비서를 만든다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 준비

오늘 만드는 것은 **설비 일지를 대신 찾아 주는 비서**다.

「3호기 어제 불량률이 2 퍼센트 넘었어?」 처럼 물으면, 모델이 스스로 사내 기록을 조회하고
필요하면 계산까지 해서 답한다. 끝까지 가면 도구를 하나 더 붙이는 데 **세 줄**이면 된다.

어제 쓰던 `build.nvidia.com` 키를 그대로 쓴다.

1. `build.nvidia.com` 에 접속해 로그인한다
2. 아무 모델이나 열고 **Get API Key** 를 누른다
3. `nvapi-` 로 시작하는 키를 복사해 아래 셀을 실행한 뒤 입력창에 붙여 넣는다

In [ ]:
# 키는 화면에 안 찍히게 받는다. 붙여 넣고 Enter 를 누르면 된다.
import getpass, json, urllib.request
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')
print('키 길이', len(KEY))     # 60~80 정도면 제대로 들어간 것이다

In [ ]:
# 모델에 대화를 통째로 보내는 함수. 실패해도 노트북이 멈추지 않게 [실패] 를 돌려준다.
URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def chat(messages, tools=None, n=400, temp=0):
    body = {'model': MODEL, 'max_tokens': n, 'temperature': temp,
            'messages': messages}
    if tools:                       # 도구 목록은 있을 때만 같이 보낸다
        body['tools'] = tools
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=120) as f:
            return json.load(f)['choices'][0]['message']
    except Exception as e:
        return {'role': 'assistant', 'content': '[실패] %s' % str(e)[:80]}

In [ ]:
# 한 문장만 물어볼 때 쓰는 짧은 이름
def say(text, n=200):
    return (chat([{'role': 'user', 'content': text}], n=n).get('content') or '').strip()

print(say('한 단어로만 답하라. 대한민국의 수도는?', 10))   # '서울' 이 나오면 준비 끝이다

> `[실패]` 가 나오면 키를 잘못 붙였거나 모델이 붐비는 것이다. 셀을 다시 실행해 본다.

## 2. 도구 없이 물어보면

먼저 **모델 혼자서는 어디까지 되는지** 본다. 성격이 다른 세 가지를 물어본다.

In [ ]:
# 세 가지를 그냥 물어본다
for q in ['17 곱하기 24 는 얼마인가? 숫자만 답하라.',
          '오늘 날짜는? 날짜만 답하라.',
          '3호기의 어제 불량률은 몇 퍼센트인가?']:
    print('Q', q)
    print('A', say(q, 60).replace('\n', ' ')[:100])
    print()

셋 다 **자신 있게** 답이 나오는데, 성격은 전부 다르다.

| 질문 | 왜 안 되나 |
|---|---|
| 17 × 24 | 자릿수를 맞춰 곱하는 구조가 모델 안에 없다 |
| 오늘 날짜 | 학습이 끝난 뒤의 일이라 알 수가 없다 |
| 3호기 불량률 | 사내 기록이라 배운 적이 없다 |

**틀린 답과 맞는 답이 똑같은 말투로 나온다.** 이것이 도구를 붙이는 이유다.

## 3. 계산 하나는 프롬프트로도 나아진다

도구로 넘어가기 전에, **프롬프트만으로 어디까지 되는지** 먼저 본다.
한 줄씩 쓰게 하면 어려운 곱셈이 쉬운 덧셈으로 갈린다.

In [ ]:
# 그냥 물었을 때와 자리별로 나누게 했을 때
print('[그냥]      ', say('17 곱하기 24 는? 숫자만 답하라.', 30).replace('\n', ' '))
print('[자리별로]  ', say('17 곱하기 24 를 자리별로 나눠 한 줄씩 계산하고 마지막 줄에 답만 써라.', 200)
      .replace('\n', ' / ')[:160])

자릿수가 늘면 다시 틀린다. 더 곤란한 것은 **맞았는지 확인할 방법이 없다**는 점이다.
그래서 정확한 값은 프롬프트로 버티지 않고 **계산기에 넘긴다**.

> **실습문제 1.** 곱하는 두 수를 **네 자리 × 세 자리**로 키워 다시 돌려 본다.
> `___` 자리에 `4271 곱하기 386` 처럼 넣으면 된다. 아래 `check` 가 정답을 같이 찍어 준다.

In [ ]:
# 자릿수를 키우면 같은 방법이 계속 통하는지 보는 것이다
q = '4271 곱하기 386 을 자리별로 나눠 한 줄씩 계산하고 마지막 줄에 답만 써라.'

print(say(q, 300))
print('정답', 4271 * 386)

## 4. 도구를 만들어 준다

도구는 특별한 것이 아니다. **평범한 파이썬 함수**다.
지금은 사내 기록 대신 표 하나를 코드 안에 넣어 두고 쓴다. 현업에서는 이 자리에 사내 DB 조회나 엑셀 읽기가 들어간다.

In [ ]:
# 어제 하루치 설비 일지 — 현업에서는 이 자리가 DB 조회나 엑셀 읽기가 된다
LOG = {
    '1호기': {'라인': 'A', '생산': 1240, '불량': 30, '근무조': '주간'},
    '2호기': {'라인': 'A', '생산':  980, '불량': 11, '근무조': '야간'},
    '3호기': {'라인': 'B', '생산': 1530, '불량': 28, '근무조': '주간'},
    '4호기': {'라인': 'B', '생산':  760, '불량': 24, '근무조': '야간'},
}
for k, v in LOG.items():
    print('%s  %s라인  생산 %5d  불량 %3d' % (k, v['라인'], v['생산'], v['불량']))

In [ ]:
# 함수 셋 — 계산기 · 오늘 날짜 · 설비 조회
import datetime

def calc(expr):
    """산술식 하나를 계산해 문자열로 돌려준다"""
    return str(eval(expr, {'__builtins__': {}}, {}))

def today():
    """오늘 날짜"""
    return datetime.date.today().isoformat()

def machine_info(machine):
    """설비 한 대의 어제 기록. 없는 이름이면 쓸 수 있는 이름을 알려 준다."""
    v = LOG.get(machine)
    if v is None:
        return '그런 설비는 없다. 쓸 수 있는 이름: ' + ', '.join(LOG)
    return '%s: %s라인, 생산 %d개, 불량 %d개, 근무조 %s' % (
        machine, v['라인'], v['생산'], v['불량'], v['근무조'])

FUNCS = {'calc': calc, 'today': today, 'machine_info': machine_info}
print(calc('17*24'))
print(today())
print(machine_info('3호기'))
print(machine_info('9호기'))     # 없는 이름을 넣으면 이렇게 알려 준다

모델은 이 함수를 **볼 수 없다**. 이름 · 설명 · 인자 모양만 글로 건네받는다.
그 설명이 곧 모델용 프롬프트다. 애매하게 적으면 도구를 안 부르거나 엉뚱하게 부른다.

In [ ]:
# 모델에게 건네는 도구 목록. spec() 은 매번 같은 모양을 찍어 주는 짧은 도우미다.
def spec(name, desc, props, required):
    return {'type': 'function', 'function': {
        'name': name, 'description': desc,
        'parameters': {'type': 'object', 'properties': props, 'required': required}}}

TOOLS = [
    spec('calc', '산술식을 계산한다. 나눗셈·퍼센트처럼 정확한 값이 필요할 때 쓴다.',
         {'expr': {'type': 'string', 'description': '파이썬 산술식. 예: 28/1530*100'}}, ['expr']),
    spec('today', '오늘 날짜를 YYYY-MM-DD 로 돌려준다.', {}, []),
    spec('machine_info', '설비 한 대의 어제 생산량·불량 수·라인·근무조를 사내 일지에서 찾아 돌려준다.',
         {'machine': {'type': 'string', 'description': '설비 이름. 예: 3호기'}}, ['machine']),
]
print(json.dumps(TOOLS[2], ensure_ascii=False, indent=1))

## 5. 모델이 도구를 고른다

도구 목록을 같이 보내면 무엇이 달라지는지 본다.

In [ ]:
# 도구 목록을 같이 보내면 답 대신 '무엇을 부를지' 가 돌아온다
m = chat([{'role': 'user', 'content': '3호기 어제 불량률은?'}], TOOLS, 200)
print('내용     ', m.get('content'))
print('도구 호출', m.get('tool_calls'))

`content` 가 비고 `tool_calls` 가 찬다. **모델이 고른 것은 답이 아니라 도구**다.
실제로 부르는 것은 우리 쪽 코드다. 결과를 다시 넣어 줘야 비로소 답이 나온다.

> `tool_calls` 가 계속 `None` 이면 그 모델이 도구 호출을 안 받는 것이다.
> 위 준비 셀의 `MODEL` 을 `meta/llama-3.3-70b-instruct` 로 바꿔 다시 실행한다.

## 6. 루프 — 판단 · 행동 · 관찰

도구를 부르고, 결과를 되먹이고, 다시 묻는다. **더 부를 것이 없을 때까지** 도는 것이 에이전트다.
아래가 그 전부다. 열 줄 남짓이고, 오늘 뒤에 나오는 것들은 전부 이 함수 주변에 붙는다.

In [ ]:
# 에이전트 본체. 마지막 대화는 LAST 에 남겨 두었다가 9절에서 다시 본다.
SYSTEM = '너는 공정 데이터 비서다. 필요하면 도구를 부르고, 모르면 모른다고 답한다.'
LAST = []

def run_agent(question, system=SYSTEM, max_steps=5, log=True):
    global LAST
    messages = [{'role': 'system', 'content': system},
                {'role': 'user', 'content': question}]
    for step in range(max_steps):
        m = chat(messages, TOOLS, 500)           # ① 판단 — 부를까, 답할까
        messages.append(m)
        calls = m.get('tool_calls') or []
        if not calls:                            # 부를 것이 없으면 그것이 답이다
            LAST = messages
            return m.get('content') or ''
        for c in calls:                          # ② 행동 — 고른 도구를 실행
            name = c['function']['name']
            args = json.loads(c['function']['arguments'] or '{}')
            out = FUNCS[name](**args)
            if log:
                print('  [도구] %s(%s) -> %s' % (name, args, out))
            messages.append({'role': 'tool', 'tool_call_id': c['id'],
                             'content': out})    # ③ 관찰 — 결과를 대화에 되먹인다
    LAST = messages
    return '[한도] %d번 안에 못 끝냈다' % max_steps

In [ ]:
# 도구가 필요한 질문
print(run_agent('3호기 어제 불량률은 몇 퍼센트야?'))

`[도구]` 줄이 실제로 부른 기록이다. 조회로 숫자를 가져오고, 계산기로 퍼센트를 냈다면 두 줄이 찍힌다.

In [ ]:
# 도구가 필요 없는 질문 — [도구] 줄이 안 찍힌다
print(run_agent('안녕? 너는 무슨 일을 하니?'))

좋은 에이전트는 **도구를 안 쓸 때도 안다**. 상식 질문까지 도구를 부르면 느리고 비싸진다.

## 7. 멀티스텝 — 앞 결과가 있어야 다음을 부른다

한 번에 안 끝나는 질문을 준다. 앞 도구의 결과를 봐야 다음 도구를 정할 수 있는 것들이다.

In [ ]:
# 조회 → 비교
print(run_agent('3호기 어제 불량률이 2 퍼센트보다 높았어?'))

In [ ]:
# 조회 네 번 → 집계
print(run_agent('1호기부터 4호기까지 어제 불량률을 모두 구해서 가장 높은 설비를 알려줘'))

`[도구]` 줄이 **두 번 이상** 찍히면 멀티스텝이다. 앞 결과를 보고 다음 도구를 정했다는 뜻이다.
이 부분이 「미리 순서를 정해 둔 코드」와 갈리는 지점이다.

도구가 **쓸 수 있는 이름을 알려 주는 에러**를 돌려주면, 모델은 그걸 읽고 다시 고른다.
`KeyError` 로 죽는 도구였다면 여기서 루프가 끝났을 것이다. **에러 문구가 곧 다음 행동의 힌트**다.

> **실습문제 2.** **없는 설비 이름**을 넣어 물어본다. 에이전트가 어떻게 빠져나오는지 본다.
> 쓸 수 있는 이름은 1호기 · 2호기 · 3호기 · 4호기 뿐이다. `7호기` 처럼 없는 것을 넣어 본다.

In [ ]:
# 도구가 던지는 에러 문구가 다음 행동을 어떻게 바꾸는지 보는 것이다
ans = run_agent('7호기의 어제 불량률을 알려줘')

print(ans)

## 8. 안전장치

루프는 스스로 멈추지 않을 수 있다. 그래서 **한도**를 같이 만든다.
`max_steps` 를 줄이면 도중에 끊긴다는 것을 먼저 확인한다.

In [ ]:
# 한도를 1로 줄이면 도구를 한 번 부르고 끝난다
print(run_agent('1호기부터 4호기까지 평균 불량률을 알려줘', max_steps=1))

현업에서는 여기에 두 가지를 더 붙인다.
**되돌릴 수 없는 도구**(삭제 · 발주 · 메일 발송)는 실행 전에 사람에게 묻고,
**부른 도구와 인자를 전부 로그로 남긴다**. 안 보이면 못 고친다.

## 9. 컨텍스트 — 대화가 얼마나 커지는가

모델은 지난 대화를 가지고 있지 않다. **매 호출마다 목록 전체를 다시 보낸다.**
그래서 도구가 돌려준 것이 쌓이면 보내는 양이 계속 커진다. 방금 대화로 직접 세어 본다.

In [ ]:
# 방금 대화에 무엇이 들어 있는지 본다
for m in LAST:
    print('%-9s %s' % (m['role'], str(m.get('content'))[:70]))
print()
print('메시지 %d개 · 글자 %d자' % (len(LAST), len(json.dumps(LAST, ensure_ascii=False))))

In [ ]:
# 도구를 많이 부르는 질문일수록 커진다
run_agent('1호기부터 4호기까지 불량률을 모두 구해서 라인별로 정리해줘', log=False)
print('메시지 %d개 · 글자 %d자' % (len(LAST), len(json.dumps(LAST, ensure_ascii=False))))

늘어나는 것은 사람이 친 말이 아니라 **도구가 돌려준 것**이다. 줄일 자리도 거기다.

### 줄이는 법 — 요약해서 넘긴다

In [ ]:
# 대화를 글로 펼쳐 요약을 받는다
def flatten(messages):
    return '\n'.join('%s: %s' % (m['role'], str(m.get('content'))[:200])
                     for m in messages)

In [ ]:
# 정한 것과 못 푼 것만 남긴다
brief = say('아래 대화를 세 줄로 요약하라. 정한 것과 아직 못 푼 것만 남기고 중복은 버려라.\n\n'
            + flatten(LAST), 300)
print(brief)
print()
print('원본 %d자 -> 요약 %d자' % (len(json.dumps(LAST, ensure_ascii=False)), len(brief)))

요약은 **되돌릴 수 없다**. 무엇을 남길지 미리 정해 두지 않으면 필요한 것부터 사라진다.
그래서 실무에서는 「정한 것 · 못 푼 것 · 파일 경로」처럼 **남길 항목을 먼저 정해 두고** 요약시킨다.

## 10. 내 업무로

여기서부터는 각자 자기 업무로 바꾼다. **루프 코드는 손대지 않는다.**
바꾸는 것은 시스템 프롬프트 한 줄, 데이터, 도구 설명뿐이다.

**도구를 늘려도 루프는 그대로다.** 바뀌는 것은 목록과 설명뿐이다.
그래서 에이전트를 키우는 일은 코드를 늘리는 일이 아니라 **도구를 정리하는 일**이 된다.

### 사내에 붙일 때 챙길 것

| 챙길 것 | 왜 |
|---|---|
| 데이터는 도구 안에 둔다 | 표를 프롬프트에 통째로 넣으면 그대로 반출이다 |
| 도구 하나는 일 하나만 | 여러 일을 하면 모델이 오용한다 |
| 에러는 고칠 방법까지 | 모델이 그 문구를 읽고 다시 고른다 |
| 되돌릴 수 없는 일은 확인 | 삭제 · 발주 · 발송은 사람에게 묻는다 |
| 부른 도구와 인자를 로그로 | 안 보이면 못 고친다 |